<a href="https://colab.research.google.com/github/Toshikiru/SyllaBot-A-RAG-Based-Course-Syllabus-and-Study-Guide-Assistant/blob/main/syllabot_embeddings_github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 SyllaBot — Checkpoint 1: Embedding Generation Demo
**IS Professional Elective #4 — Generative AI Systems**  
**Group: Sugar Group | A.Y. 2026–2027**

This notebook covers **Deliverable #4** of Checkpoint 1:
- Pull cleaned `.txt` files directly from a GitHub repo folder
- Split text into chunks for embedding
- Generate embeddings using HuggingFace `all-MiniLM-L6-v2`
- Visualize and explain the embeddings
- Save embeddings for use in Checkpoint 2

## Step 1 — Install Required Libraries

In [ ]:
!pip install sentence-transformers numpy pandas matplotlib scikit-learn requests -q

## Step 2 — Fetch Cleaned `.txt` Files from GitHub
Instead of manually uploading files each run, this pulls every `.txt` file straight from a folder in your GitHub repo using the GitHub Contents API.

**⚠️ Edit the placeholders below** with your actual repo details before running.

- `GITHUB_OWNER` / `GITHUB_REPO` — e.g. for `https://github.com/kei-dev/syllabot`, owner is `kei-dev`, repo is `syllabot`
- `GITHUB_FOLDER` — path to the folder holding the cleaned `.txt` files (e.g. `syllabi_cleaned`)
- `GITHUB_BRANCH` — usually `main` or `master`
- `GITHUB_TOKEN` — optional. Unauthenticated requests are capped at 60/hour; a [personal access token](https://github.com/settings/tokens) raises that to 5000/hour. Leave as `None` if your repo is public and small.

In [ ]:
import os
import requests


GITHUB_OWNER = "Toshikiru"
GITHUB_REPO = "SyllaBot-A-RAG-Based-Course-Syllabus-and-Study-Guide-Assistant"
GITHUB_FOLDER = "Checkpoint1/data_source_raw/syllabu_cleaned"
GITHUB_BRANCH = "main"
GITHUB_TOKEN = None  # e.g. "ghp_xxx..." — optional, only needed for private repos or to avoid rate limits
# ------------------

os.makedirs('syllabi_cleaned', exist_ok=True)

headers = {'Accept': 'application/vnd.github+json'}
if GITHUB_TOKEN:
    headers['Authorization'] = f'Bearer {GITHUB_TOKEN}'

api_url = (
    f'https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}'
    f'/contents/{GITHUB_FOLDER}?ref={GITHUB_BRANCH}'
)

print(f'📡 Requesting file list from: {api_url}')
response = requests.get(api_url, headers=headers, timeout=30)

if response.status_code == 404:
    raise RuntimeError(
        'GitHub returned 404 — double check GITHUB_OWNER, GITHUB_REPO, GITHUB_FOLDER, '
        'and GITHUB_BRANCH are correct and the repo/folder is public (or GITHUB_TOKEN is set).'
    )
if response.status_code == 403 and 'rate limit' in response.text.lower():
    raise RuntimeError(
        'GitHub API rate limit exceeded. Set GITHUB_TOKEN to a personal access token and re-run.'
    )
response.raise_for_status()

listing = response.json()
if not isinstance(listing, list):
    raise RuntimeError(
        f'Expected a folder listing but got something else. GitHub said: {listing}'
    )

txt_files = [f for f in listing if f.get('type') == 'file' and f['name'].endswith('.txt')]
print(f'📁 Found {len(txt_files)} .txt files in {GITHUB_OWNER}/{GITHUB_REPO}/{GITHUB_FOLDER}')

if not txt_files:
    print('⚠️  No .txt files found. Check the folder path and branch name.')

downloaded = 0
for f in txt_files:
    file_resp = requests.get(f['download_url'], headers=headers, timeout=30)
    file_resp.raise_for_status()
    local_path = os.path.join('syllabi_cleaned', f['name'])
    with open(local_path, 'w', encoding='utf-8') as out:
        out.write(file_resp.text)
    downloaded += 1
    print(f'  ✅ Downloaded: {f["name"]}')

print(f'\n📁 Total files downloaded: {downloaded}')

📡 Requesting file list from: https://api.github.com/repos/Toshikiru/SyllaBot-A-RAG-Based-Course-Syllabus-and-Study-Guide-Assistant/contents/Checkpoint1/data_source_raw/syllabu_cleaned?ref=main
📁 Found 8 .txt files in Toshikiru/SyllaBot-A-RAG-Based-Course-Syllabus-and-Study-Guide-Assistant/Checkpoint1/data_source_raw/syllabu_cleaned
  ✅ Downloaded: 01_GenAI_Systems_Syllabus.txt
  ✅ Downloaded: 02_Business_Analytics_Syllabus.txt
  ✅ Downloaded: 03_Information_Management_Syllabus.txt
  ✅ Downloaded: 04_Systems_Analysis_Design_Syllabus.txt
  ✅ Downloaded: 05_Thesis_Writing_Syllabus.txt
  ✅ Downloaded: 06_Academic_Calendar_and_Policies.txt
  ✅ Downloaded: 07_IT_Ethics_Syllabus.txt
  ✅ Downloaded: 08_Software_Engineering_Syllabus.txt

📁 Total files downloaded: 8


## Step 3 — Load All Cleaned Text Files

In [ ]:
documents = {}  # {filename: full_text}

for filename in sorted(os.listdir('syllabi_cleaned')):
    if filename.endswith('.txt'):
        with open(f'syllabi_cleaned/{filename}', 'r', encoding='utf-8') as f:
            text = f.read().strip()
            if text:
                documents[filename] = text
                print(f'  📄 Loaded: {filename} ({len(text):,} characters)')
            else:
                print(f'  ⚠️  Skipped empty file: {filename}')

print(f'\n✅ Total documents loaded: {len(documents)}')

if not documents:
    raise RuntimeError(
        'No documents loaded. Check that Step 2 actually downloaded .txt files into syllabi_cleaned/.'
    )

  📄 Loaded: 01_GenAI_Systems_Syllabus.txt (2,562 characters)
  📄 Loaded: 02_Business_Analytics_Syllabus.txt (2,025 characters)
  📄 Loaded: 03_Information_Management_Syllabus.txt (2,686 characters)
  📄 Loaded: 04_Systems_Analysis_Design_Syllabus.txt (2,713 characters)
  📄 Loaded: 05_Thesis_Writing_Syllabus.txt (3,174 characters)
  📄 Loaded: 06_Academic_Calendar_and_Policies.txt (3,791 characters)
  📄 Loaded: 07_IT_Ethics_Syllabus.txt (3,566 characters)
  📄 Loaded: 08_Software_Engineering_Syllabus.txt (3,348 characters)

✅ Total documents loaded: 8


## Step 4 — Split Text into Chunks
We cannot embed an entire document at once — it would be too long.
We split each document into smaller overlapping chunks of ~500 characters.

**Why overlap?** So that sentences at chunk boundaries are not cut off and lost.

In [ ]:
def split_into_chunks(text, chunk_size=500, overlap=50):
    """
    Split text into overlapping chunks.
    - chunk_size: number of characters per chunk
    - overlap: number of characters shared between consecutive chunks
    """
    if overlap >= chunk_size:
        raise ValueError('overlap must be smaller than chunk_size, or chunking never advances.')

    chunks = []
    start = 0
    text_len = len(text)
    while start < text_len:
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

# Apply chunking to all documents
all_chunks = []       # list of chunk texts
chunk_metadata = []   # list of {source, chunk_index} for each chunk

for filename, text in documents.items():
    chunks = split_into_chunks(text, chunk_size=500, overlap=50)
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        chunk_metadata.append({'source': filename, 'chunk_index': i})
    print(f'  🔪 {filename}: {len(chunks)} chunks')

print(f'\n✅ Total chunks created: {len(all_chunks)}')

if not all_chunks:
    raise RuntimeError('No chunks were created — check that the loaded documents have content.')

  🔪 01_GenAI_Systems_Syllabus.txt: 6 chunks
  🔪 02_Business_Analytics_Syllabus.txt: 5 chunks
  🔪 03_Information_Management_Syllabus.txt: 6 chunks
  🔪 04_Systems_Analysis_Design_Syllabus.txt: 7 chunks
  🔪 05_Thesis_Writing_Syllabus.txt: 8 chunks
  🔪 06_Academic_Calendar_and_Policies.txt: 9 chunks
  🔪 07_IT_Ethics_Syllabus.txt: 8 chunks
  🔪 08_Software_Engineering_Syllabus.txt: 8 chunks

✅ Total chunks created: 57


## Step 5 — Preview a Chunk (Before Embedding)

In [ ]:
print('=' * 60)
print('📋 SAMPLE CHUNK (what gets fed into the embedding model):')
print('=' * 60)
print(f'Source: {chunk_metadata[0]["source"]}')
print(f'Chunk index: {chunk_metadata[0]["chunk_index"]}')
print(f'Character length: {len(all_chunks[0])}')
print()
print(all_chunks[0])

📋 SAMPLE CHUNK (what gets fed into the embedding model):
Source: 01_GenAI_Systems_Syllabus.txt
Chunk index: 0
Character length: 500

course syllabus
is professional elective 4 generative ai systems
instructor: jessie a. melendres 1st semester, a.y. 20262027 units: 3 schedule: tth 1:002:30 pm
1. course description
this course introduces students to the principles and applications of generative ai systems. topics include transformer architecture, large language models (llms), prompt engineering, retrieval-augmented generation (rag), vector databases, fine-tuning techniques such as lora and qlora, and llmops deployment practices


## Step 6 — Load the Embedding Model
We use `all-MiniLM-L6-v2` from HuggingFace.

### Why this model?
- ✅ **Free** — no API key or cost needed
- ✅ **Lightweight** — only 80MB, runs fast even on Colab CPU
- ✅ **High quality** — produces 384-dimension embeddings optimized for semantic similarity search
- ✅ **Widely used** — standard choice for RAG pipelines in production

In [ ]:
from sentence_transformers import SentenceTransformer

print('⏳ Loading embedding model: all-MiniLM-L6-v2...')
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('✅ Model loaded!')
print(f'📏 Embedding dimensions: {model.get_sentence_embedding_dimension()}')

⏳ Loading embedding model: all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded!
📏 Embedding dimensions: 384


/tmp/ipykernel_677/808563050.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'📏 Embedding dimensions: {model.get_sentence_embedding_dimension()}')


## Step 7 — Generate Embeddings
Convert all text chunks into embedding vectors.
Each chunk becomes a list of 384 numbers that capture its meaning.

In [ ]:
import numpy as np

print(f'⏳ Generating embeddings for {len(all_chunks)} chunks...')
print('(This may take 1–2 minutes on CPU)\n')

embeddings = model.encode(
    all_chunks,
    show_progress_bar=True,
    batch_size=32
)

print(f'\n✅ Embeddings generated!')
print(f'📊 Shape: {embeddings.shape}')  # (num_chunks, 384)
print(f'   → {embeddings.shape[0]} chunks × {embeddings.shape[1]} dimensions each')

⏳ Generating embeddings for 57 chunks...
(This may take 1–2 minutes on CPU)



Batches:   0%|          | 0/2 [00:00<?, ?it/s]


✅ Embeddings generated!
📊 Shape: (57, 384)
   → 57 chunks × 384 dimensions each


## Step 8 — What Does an Embedding Look Like?
This shows the actual embedding vector of the first chunk.

In [ ]:
print('=' * 60)
print('🔢 SAMPLE EMBEDDING VECTOR (first 20 of 384 dimensions):')
print('=' * 60)
print(f'\nSource text (first 100 chars):')
print(f'  "{all_chunks[0][:100]}..."')
print(f'\nEmbedding vector (first 20 values):')
print(f'  {embeddings[0][:20].tolist()}')
print(f'\nFull vector shape: {embeddings[0].shape}')
print(f'Value range: min={embeddings[0].min():.4f}, max={embeddings[0].max():.4f}')
print()
print('💡 These numbers represent the MEANING of the text in 384-dimensional space.')
print('   Similar texts will have vectors that point in similar directions.')

🔢 SAMPLE EMBEDDING VECTOR (first 20 of 384 dimensions):

Source text (first 100 chars):
  "course syllabus
is professional elective 4 generative ai systems
instructor: jessie a. melendres 1st..."

Embedding vector (first 20 values):
  [-0.03811265528202057, -0.06275511533021927, 0.03969356045126915, -0.07304280251264572, -0.06320001184940338, -0.018426021561026573, -0.019182000309228897, 0.011368775740265846, -0.043317023664712906, 0.023453164845705032, -0.05603078380227089, -0.043846845626831055, 0.04409467801451683, -0.048846594989299774, -0.0018837348325178027, 0.05154690518975258, 0.0014963269932195544, 0.008785083889961243, 0.05511542037129402, -0.07866454124450684]

Full vector shape: (384,)
Value range: min=-0.1103, max=0.1487

💡 These numbers represent the MEANING of the text in 384-dimensional space.
   Similar texts will have vectors that point in similar directions.


## Step 9 — Semantic Similarity Demo
We test whether similar syllabus content produces similar embeddings by computing cosine similarity.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Test sentences
test_queries = [
    'What is the grading system for this course?',
    'When is the midterm exam?',
    'What are the attendance policies?',
]

print('🔍 SEMANTIC SIMILARITY DEMO')
print('Finding the most relevant chunk for each query...\n')

for query in test_queries:
    # Embed the query
    query_embedding = model.encode([query])

    # Compute cosine similarity against all chunks
    similarities = cosine_similarity(query_embedding, embeddings)[0]

    # Get top result
    top_idx = int(similarities.argmax())
    top_score = similarities[top_idx]
    top_source = chunk_metadata[top_idx]['source']

    print(f'Query : "{query}"')
    print(f'Source: {top_source}')
    print(f'Score : {top_score:.4f}')
    print(f'Chunk : "{all_chunks[top_idx][:200]}..."')
    print('-' * 60)

🔍 SEMANTIC SIMILARITY DEMO
Finding the most relevant chunk for each query...

Query : "What is the grading system for this course?"
Source: 08_Software_Engineering_Syllabus.txt
Score : 0.5146
Chunk : "mo
16 final examination final exam
4. grading system
component weight
quizzes / participation 10%
laboratory exercises 20%
performance tasks 20%
group software project 20%
midterm exam 15%
final exam ..."
------------------------------------------------------------
Query : "When is the midterm exam?"
Source: 06_Academic_Calendar_and_Policies.txt
Score : 0.5533
Chunk : "academic calendar and student policies 1st semester, a.y.
academic calendar
period dates remarks
enrollment period july 28 august 1, 2026 regular enrollment
late enrollment august 45, 2026 with late f..."
------------------------------------------------------------
Query : "What are the attendance policies?"
Source: 01_GenAI_Systems_Syllabus.txt
Score : 0.5464
Chunk : "5. project checkpoints
6. course policies
attendance: s

## Step 10 — Save Embeddings and Metadata
Save everything so it can be loaded in Checkpoint 2 (Vector Database Setup).

In [ ]:
import json

os.makedirs('embeddings_output', exist_ok=True)

# Save embeddings as numpy array
np.save('embeddings_output/syllabot_embeddings.npy', embeddings)

# Save chunks and metadata as JSON
output_data = [
    {
        'chunk_id': i,
        'source': chunk_metadata[i]['source'],
        'chunk_index': chunk_metadata[i]['chunk_index'],
        'text': all_chunks[i]
    }
    for i in range(len(all_chunks))
]

with open('embeddings_output/syllabot_chunks.json', 'w', encoding='utf-8') as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print('✅ Files saved to /embeddings_output/')
print(f'   syllabot_embeddings.npy — {embeddings.shape[0]} vectors × {embeddings.shape[1]} dims')
print(f'   syllabot_chunks.json   — {len(output_data)} chunk records')
print('\n📌 Keep these files — they are the input for Checkpoint 2 (Vector Database).')

✅ Files saved to /embeddings_output/
   syllabot_embeddings.npy — 57 vectors × 384 dims
   syllabot_chunks.json   — 57 chunk records

📌 Keep these files — they are the input for Checkpoint 2 (Vector Database).


## Step 11 — Download Output Files
This only works in Google Colab (it uses `google.colab.files`). If you're running locally or in another environment, just grab the files from the `embeddings_output/` folder directly.

In [ ]:
try:
    from google.colab import files
    print('⬇️  Downloading embedding output files...')
    files.download('embeddings_output/syllabot_embeddings.npy')
    files.download('embeddings_output/syllabot_chunks.json')
    print('✅ Download started!')
except ImportError:
    print('ℹ️  Not running in Google Colab — skip this step.')
    print('   Your files are already saved locally in embeddings_output/.')

⬇️  Downloading embedding output files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started!


## Step 12 — Summary

In [ ]:
print('=' * 60)
print('📋 CHECKPOINT 1 — EMBEDDING GENERATION SUMMARY')
print('=' * 60)
print(f'Source repo     : {GITHUB_OWNER}/{GITHUB_REPO}/{GITHUB_FOLDER} ({GITHUB_BRANCH})')
print(f'Embedding model : sentence-transformers/all-MiniLM-L6-v2')
print(f'Model size      : ~80MB (lightweight, CPU-friendly)')
print(f'Dimensions      : 384 per chunk')
print(f'Chunk size      : 500 characters with 50-character overlap')
print(f'Documents       : {len(documents)}')
print(f'Total chunks    : {len(all_chunks)}')
print(f'Total embeddings: {embeddings.shape[0]} vectors')
print()
print('Why all-MiniLM-L6-v2?')
print('  - Free, no API key required')
print('  - Fast on CPU, ideal for Colab')
print('  - 384-dim vectors: compact but semantically rich')
print('  - Trained specifically for semantic similarity tasks')
print('  - Standard choice for RAG pipelines in production')
print()
print('Next step → Checkpoint 2: Load these embeddings into')
print('a vector database (ChromaDB or FAISS) for similarity search.')

📋 CHECKPOINT 1 — EMBEDDING GENERATION SUMMARY
Source repo     : Toshikiru/SyllaBot-A-RAG-Based-Course-Syllabus-and-Study-Guide-Assistant/Checkpoint1/data_source_raw/syllabu_cleaned (main)
Embedding model : sentence-transformers/all-MiniLM-L6-v2
Model size      : ~80MB (lightweight, CPU-friendly)
Dimensions      : 384 per chunk
Chunk size      : 500 characters with 50-character overlap
Documents       : 8
Total chunks    : 57
Total embeddings: 57 vectors

Why all-MiniLM-L6-v2?
  - Free, no API key required
  - Fast on CPU, ideal for Colab
  - 384-dim vectors: compact but semantically rich
  - Trained specifically for semantic similarity tasks
  - Standard choice for RAG pipelines in production

Next step → Checkpoint 2: Load these embeddings into
a vector database (ChromaDB or FAISS) for similarity search.
